In [7]:
import numpy as np

In [8]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense

In [9]:
batch_size = 64
epochs = 100
latent_dim = 256 # Vector size that repersentive one Token
num_samples = 10000
data_path = '/content/En_Fr.txt'

In [10]:
input_texts = [] # For English
target_texts = [] # For France

In [11]:
input_characters = set()
target_characters = set()

In [12]:
with open(data_path, 'r', encoding='utf-8') as f:
  lines = f.read().split('\n')

In [13]:
lines

['Go.\tVa !',
 'Hi.\tSalut !',
 'Run!\tCours\u202f!',
 'Run!\tCourez\u202f!',
 'Who?\tQui ?',
 'Wow!\tÇa alors\u202f!',
 'Fire!\tAu feu !',
 "Help!\tÀ l'aide\u202f!",
 'Jump.\tSaute.',
 'Stop!\tÇa suffit\u202f!',
 'Stop!\tStop\u202f!',
 'Stop!\tArrête-toi !',
 'Wait!\tAttends !',
 'Wait!\tAttendez !',
 'Go on.\tPoursuis.',
 'Go on.\tContinuez.',
 'Go on.\tPoursuivez.',
 'Hello!\tBonjour !',
 'Hello!\tSalut !',
 'I see.\tJe comprends.',
 "I try.\tJ'essaye.",
 "I won!\tJ'ai gagné !",
 "I won!\tJe l'ai emporté !",
 'I won.\tJ’ai gagné.',
 'Oh no!\tOh non !',
 'Attack!\tAttaque !',
 'Attack!\tAttaquez !',
 'Cheers!\tSanté !',
 'Cheers!\tÀ votre santé !',
 'Cheers!\tMerci !',
 'Cheers!\tTchin-tchin !',
 'Get up.\tLève-toi.',
 'Go now.\tVa, maintenant.',
 'Go now.\tAllez-y maintenant.',
 'Go now.\tVas-y maintenant.',
 "Got it!\tJ'ai pigé !",
 'Got it!\tCompris !',
 'Got it?\tPigé\u202f?',
 'Got it?\tCompris\u202f?',
 "Got it?\tT'as capté\u202f?",
 'Hop in.\tMonte.',
 'Hop in.\tMontez.',
 'Hu

In [14]:
lines[:1]

['Go.\tVa !']

In [15]:
for line in lines[: min(num_samples, len(lines) - 1)]:
  input_text, target_text = line.split('\t')
  target_text = '\t' + target_text + '\n'

  input_texts.append(input_text)
  target_texts.append(target_text)

  for char in input_text:
    if char not in input_characters:
      input_characters.add(char)

  for char in target_text:
    if char not in target_characters:
      target_characters.add(char)


In [16]:
print(input_texts)

['Go.', 'Hi.', 'Run!', 'Run!', 'Who?', 'Wow!', 'Fire!', 'Help!', 'Jump.', 'Stop!', 'Stop!', 'Stop!', 'Wait!', 'Wait!', 'Go on.', 'Go on.', 'Go on.', 'Hello!', 'Hello!', 'I see.', 'I try.', 'I won!', 'I won!', 'I won.', 'Oh no!', 'Attack!', 'Attack!', 'Cheers!', 'Cheers!', 'Cheers!', 'Cheers!', 'Get up.', 'Go now.', 'Go now.', 'Go now.', 'Got it!', 'Got it!', 'Got it?', 'Got it?', 'Got it?', 'Hop in.', 'Hop in.', 'Hug me.', 'Hug me.', 'I fell.', 'I fell.', 'I know.', 'I left.', 'I left.', 'I lost.', 'I paid.', "I'm 19.", "I'm OK.", "I'm OK.", 'Listen.', 'No way!', 'No way!', 'No way!', 'No way!', 'No way!', 'No way!', 'No way!', 'No way!', 'No way!', 'Really?', 'Really?', 'Really?', 'Thanks.', 'We try.', 'We won.', 'We won.', 'We won.', 'We won.', 'Ask Tom.', 'Awesome!', 'Be calm.', 'Be calm.', 'Be calm.', 'Be cool.', 'Be fair.', 'Be fair.', 'Be fair.', 'Be fair.', 'Be fair.', 'Be fair.', 'Be kind.', 'Be nice.', 'Be nice.', 'Be nice.', 'Be nice.', 'Be nice.', 'Be nice.', 'Beat it.', 'Ca

In [17]:
print(target_texts)

['\tVa !\n', '\tSalut !\n', '\tCours\u202f!\n', '\tCourez\u202f!\n', '\tQui ?\n', '\tÇa alors\u202f!\n', '\tAu feu !\n', "\tÀ l'aide\u202f!\n", '\tSaute.\n', '\tÇa suffit\u202f!\n', '\tStop\u202f!\n', '\tArrête-toi !\n', '\tAttends !\n', '\tAttendez !\n', '\tPoursuis.\n', '\tContinuez.\n', '\tPoursuivez.\n', '\tBonjour !\n', '\tSalut !\n', '\tJe comprends.\n', "\tJ'essaye.\n", "\tJ'ai gagné !\n", "\tJe l'ai emporté !\n", '\tJ’ai gagné.\n', '\tOh non !\n', '\tAttaque !\n', '\tAttaquez !\n', '\tSanté !\n', '\tÀ votre santé !\n', '\tMerci !\n', '\tTchin-tchin !\n', '\tLève-toi.\n', '\tVa, maintenant.\n', '\tAllez-y maintenant.\n', '\tVas-y maintenant.\n', "\tJ'ai pigé !\n", '\tCompris !\n', '\tPigé\u202f?\n', '\tCompris\u202f?\n', "\tT'as capté\u202f?\n", '\tMonte.\n', '\tMontez.\n', '\tSerre-moi dans tes bras !\n', '\tSerrez-moi dans vos bras !\n', '\tJe suis tombée.\n', '\tJe suis tombé.\n', '\tJe sais.\n', '\tJe suis parti.\n', '\tJe suis partie.\n', "\tJ'ai perdu.\n", '\tJ’ai payé.\n'

In [18]:
print(input_characters)

{'E', 'j', 'I', 'N', 'Q', 'u', '8', 'L', 'i', 'A', 'F', 's', 'o', 'T', '&', '3', '5', 'a', 'Y', '%', '-', 'k', '0', 'R', '?', '1', 'p', 'z', 'v', 'w', ',', 't', 'U', '7', 'f', "'", 'M', 'e', 'h', 'm', 'c', 'W', 'P', '.', 'K', 'x', '2', 'J', 'V', 'b', 'n', 'S', 'G', '$', 'O', 'r', 'D', '6', 'y', 'C', '!', ' ', 'B', ':', 'q', 'l', 'g', 'd', '9', 'H'}


In [19]:
print(target_characters)

{'Ç', 'I', 'Ê', '\t', '%', '1', 't', 'f', "'", 'M', 'ê', 'b', 'y', 'C', 'û', 'B', ':', 'S', 'E', 'N', 'u', 'L', 'T', 'A', 'ô', '&', 'ë', '3', '»', 'a', 'À', '«', ',', 'ï', 'h', 'x', 'c', '.', '2', 'J', '$', 'â', 'D', ' ', 'g', 'd', '9', 'j', 'î', 'Q', '8', 's', 'o', 'Y', '’', 'ù', 'è', 'p', ')', 'e', 'P', 'K', '\u2009', '\u202f', 'G', 'é', '!', 'l', 'à', 'œ', 'H', '\xa0', 'i', 'F', 'É', '5', '-', 'k', '0', '?', 'R', 'z', 'v', 'U', '\n', 'm', 'ç', 'V', 'n', 'O', 'r', '(', 'q'}


In [20]:
input_characters = sorted(list(input_characters))

In [21]:
target_characters = sorted(list(target_characters))

In [22]:
num_encoder_tokens = len(input_characters)
num_decoder_tokens  = len(target_characters)

In [23]:
max_encoder_seq_length = max([len(text) for text in input_texts])
max_Decoder_seq_length = max([len(text) for text in target_texts])

In [24]:
print('Number of Samples >>', len(input_texts))
print('Number of Unique Input Tokens >>', num_encoder_tokens)
print('Number of Unique Output Tokens >>', num_decoder_tokens)
print('Max Sequence Length of Inputs >>', max_encoder_seq_length)
print('Max Sequence Length of Output >>', max_Decoder_seq_length)

Number of Samples >> 10000
Number of Unique Input Tokens >> 70
Number of Unique Output Tokens >> 93
Max Sequence Length of Inputs >> 16
Max Sequence Length of Output >> 59


In [25]:
input_token_index = dict(
    [(char, i) for i, char in enumerate(input_characters)]
)

In [26]:
output_token_index = dict(
    [(char, i) for i, char in enumerate(target_characters)]
)

In [27]:
print(input_token_index)

{' ': 0, '!': 1, '$': 2, '%': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '0': 9, '1': 10, '2': 11, '3': 12, '5': 13, '6': 14, '7': 15, '8': 16, '9': 17, ':': 18, '?': 19, 'A': 20, 'B': 21, 'C': 22, 'D': 23, 'E': 24, 'F': 25, 'G': 26, 'H': 27, 'I': 28, 'J': 29, 'K': 30, 'L': 31, 'M': 32, 'N': 33, 'O': 34, 'P': 35, 'Q': 36, 'R': 37, 'S': 38, 'T': 39, 'U': 40, 'V': 41, 'W': 42, 'Y': 43, 'a': 44, 'b': 45, 'c': 46, 'd': 47, 'e': 48, 'f': 49, 'g': 50, 'h': 51, 'i': 52, 'j': 53, 'k': 54, 'l': 55, 'm': 56, 'n': 57, 'o': 58, 'p': 59, 'q': 60, 'r': 61, 's': 62, 't': 63, 'u': 64, 'v': 65, 'w': 66, 'x': 67, 'y': 68, 'z': 69}


In [28]:
print(output_token_index)

{'\t': 0, '\n': 1, ' ': 2, '!': 3, '$': 4, '%': 5, '&': 6, "'": 7, '(': 8, ')': 9, ',': 10, '-': 11, '.': 12, '0': 13, '1': 14, '2': 15, '3': 16, '5': 17, '8': 18, '9': 19, ':': 20, '?': 21, 'A': 22, 'B': 23, 'C': 24, 'D': 25, 'E': 26, 'F': 27, 'G': 28, 'H': 29, 'I': 30, 'J': 31, 'K': 32, 'L': 33, 'M': 34, 'N': 35, 'O': 36, 'P': 37, 'Q': 38, 'R': 39, 'S': 40, 'T': 41, 'U': 42, 'V': 43, 'Y': 44, 'a': 45, 'b': 46, 'c': 47, 'd': 48, 'e': 49, 'f': 50, 'g': 51, 'h': 52, 'i': 53, 'j': 54, 'k': 55, 'l': 56, 'm': 57, 'n': 58, 'o': 59, 'p': 60, 'q': 61, 'r': 62, 's': 63, 't': 64, 'u': 65, 'v': 66, 'x': 67, 'y': 68, 'z': 69, '\xa0': 70, '«': 71, '»': 72, 'À': 73, 'Ç': 74, 'É': 75, 'Ê': 76, 'à': 77, 'â': 78, 'ç': 79, 'è': 80, 'é': 81, 'ê': 82, 'ë': 83, 'î': 84, 'ï': 85, 'ô': 86, 'ù': 87, 'û': 88, 'œ': 89, '\u2009': 90, '’': 91, '\u202f': 92}


Encoder Input

In [29]:
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype='float32')

 Decoder Input

In [30]:
Decoder_input_data = np.zeros((len(input_texts), max_Decoder_seq_length, num_decoder_tokens), dtype='float32')

Decoder Output

In [31]:
decoder_output_data = np.zeros((len(input_texts), max_Decoder_seq_length, num_decoder_tokens), dtype='float32')

In [32]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):

  # Encoding Input Text
  for t, char in enumerate(input_text):
    encoder_input_data[i, t, input_token_index[char]] = 1

  # Encoding target text and creating decoder input and output data
  for t, char in enumerate(target_text):
    if t > 0:
      decoder_output_data[i, t - 1, output_token_index[char]] = 1
    Decoder_input_data[i, t, output_token_index[char]] = 1
  Decoder_input_data[i, t + 1 : output_token_index[' ']] = 1
  decoder_output_data[i, t : output_token_index[' ']] = 1

Encoder

In [33]:
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True, return_sequences=True)

encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

Decoder

In [34]:
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_state=True, return_sequences=True)

decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [35]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [36]:
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

In [37]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 70)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 93)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │    334,848 │ input_layer[0][0] │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    358,400 │ input_layer_1[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 93)  │     23,901 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 717,149 (2.74 MB)

 Trainable params: 717,149 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.fit(
    [encoder_input_data, Decoder_input_data],
    decoder_output_data,
    batch_size=64,
    epochs=100,
    validation_split=.2
)

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.0415 - loss: 1.1786 - val_accuracy: 0.0509 - val_loss: 1.1641
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.0512 - loss: 1.0217 - val_accuracy: 0.0516 - val_loss: 1.1460
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.0534 - loss: 1.0027 - val_accuracy: 0.0533 - val_loss: 1.1354
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0556 - loss: 0.9882 - val_accuracy: 0.0547 - val_loss: 1.1338
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.0585 - loss: 0.9727 - val_accuracy: 0.0623 - val_loss: 1.1015
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0620 - loss: 0.9578 - val_accuracy: 0.0671 - val_loss: 1.0971
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.0670 - loss: 0.9512 - val_accuracy: 0.0702 - val_loss: 1.0821
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.0703 - loss: 0.9324 - 